In [2]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

C:\Users\24978\.conda\envs\llm\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [3]:
""" 初始化向量数据库 """
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(
    name="server_logs",
    metadata={"hnsw:space": "cosine"}
)

In [4]:

# 加载embedding模型。使用已有大embedding模型
model_path = 'D:/cs/projects/RAG/mooc/RAG_full_stack_course_notebooks/llm_app/gte-large-zh/'
# embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
embed_model = SentenceTransformer(model_path)

In [5]:
""" 将预处理后的日志块存入向量数据库 """
def store_log_blocks(log_blocks: list):
    # 分别存储log_id, 文档, 元数据
    ids, documents, metadatas = [], [], []

    # 对于单个日志块
    for block in log_blocks:
        # 设置日志id，起始时间+调用者的hash
        log_id = f"{block['metadata']['start_time']}-{hash(block['metadata']['services'][0])}"
        ids.append(log_id)
        documents.append(block['text_for_embedding'])
        metadatas.append({
            "request_id": block['metadata']['request_id'],
            "start_time": block['metadata']['start_time'],
            "service": block['metadata']['services'][0]
        })

    # 批量生成向量(CPU高效)
    embeddings = embed_model.encode(documents).tolist()

    # 存入向量数据库
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

In [6]:
"""
混合检索日志
:param query: 自然语言查询
:param filters: 元数据过滤
:param top_k: 返回结果数量
"""
def search_logs(query: str, filters: dict = None, top_k:int = 5):
    # 1. 生成查询向量, 用于模糊查询
    query_embedding = embed_model.encode([query]).tolist()[0]

    # 2. 构建混合查询条件
    where_conditions = []
    if filters:
        if "start_time" in filters:
            where_conditions["start_time"] = {"$gte": filters["start_time"]}
        if "service" in filters:
            where_conditions["service"] = {"$eq": filters["service"]}

    # 3. 执行检索: 元数据过滤 + 语义相似度
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_conditions, # 元数据过滤
        include=["documents", "metadatas"] # 返回原始文本和元数据
    )

    # 4. 整理结果
    output = []
    for i in range(len(results["ids"[0]])):
        output.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "score": results["distances"][0][i]
        })

    return output